# Use Langchain for RAG
Input sources to be used for the laws:
1. Laws of Cricket 2022 - latest edition
2. Interpretation of the laws - Tom Smith old 2019 edition
3. Changes to the laws from 2019 to 2022 - to aid interpretation of changed laws
4. Interpretations of specific scenarios compiled from MCC articles

Input sources for playing conditions:
1. ICC playing conditions for different game formats and gender-specific conditions
2. Any other playing conditions for local leagues

The framework is to set up three vector databases:
1. For the law document alone (the 2022 version)
2. For interpretation of the laws (Tom Smith's and other sources)
3. For the playing condtions alone

Agentic RAG to be performed based on the query to source results from the laws and the playing conditions, collating the results. Possibly a single vector database holding all of them may be sufficient and would be the first one to test.


In [1]:
#!pip install -U langchain-community

In [2]:
'''
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFaceHub
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.document_loaders import UnstructuredImageLoader
'''
import pymupdf  # PyMuPDF library
import os
import re
import shutil
import pandas as pd

In [3]:
def imageSave(images,file,page_index,image_path):
    filename = file.split(os.sep)[-1][:-4]
    for image_index, img in enumerate(images, start=1): # enumerate the image list
        xref = img[0] # get the XREF of the image
        with pymupdf.open(file) as doc:
            pix = pymupdf.Pixmap(doc, xref) # create a Pixmap

            if pix.n - pix.alpha > 3: # CMYK: convert to RGB first
                pix = pymupdf.Pixmap(pymupdf.csRGB, pix)                
    
            pix.save(image_path+os.sep+"Doc_%s-page_%s-image_%s.png" % (filename,page_index, image_index)) # save the image as png
            pix = None  


def getFilesList(pdf_path):
    files_list = []
    for root, dirs, files in os.walk(pdf_path):
        for file in files:
          files_list.append(os.path.join(root, file))

    return files_list
    
    
def extractData(pdf_path):

    save_path = 'RAG_prep'+os.sep+pdf_path.split(os.sep)[-1]
    
    files_list = getFilesList(pdf_path)


    
    for file in files_list:
        if file[-3:]=='pdf':
            with pymupdf.open(file) as doc:
                print('Document: ',doc)
                print('Number of pages: ', len(doc))
                all_text = ""
                filename = file.split(os.sep)[-1][:-4]
                for page_num in range(len(doc)):
                    page = doc.load_page(page_num)
                    text = str(page.get_text().encode("utf-8"))#.decode("utf-8")  
                    
                    #text = text.encode("utf-8")
                    text = text.encode("charmap", errors="ignore")
                    all_text += str(text)
                    # remove unwanted text
                    # further text preprocessing needs to be done later to remove special characters etc before text splitting
                    # will use a clean text function
                    
                    #tp = page.get_textpage_ocr()
                    #ocr_text = page.get_text(textpage=tp)
                    images = page.get_images()
                    tabs = page.find_tables()
                    #print('Page number:', page_num)
                    #print(text[:100])
                    #print('Images: ',images)
        
                    if images:                        
                        if (filename[:4]=='MCC_') and (page_num>0): 
                            #print("moving to next... page number", page_num)
                            continue
                            
                        dest_dir = os.path.join(save_path,'Images')
                        os.makedirs(dest_dir, exist_ok=True)
                        imageSave(images,file,page_num,dest_dir)
    
                    tab_ext = 0
                    if tabs.tables: 
                        for ta in range(len(tabs.tables)):
                            tab_ext = tabs[ta].extract()
                            dest_dir = os.path.join(save_path,'Tables')
                            os.makedirs(dest_dir, exist_ok=True)                            
                            df = pd.DataFrame(tab_ext)                            
                            dest_file = os.path.join(dest_dir,f"Doc_{filename}-page-{page_num}-table-{ta}.csv")
                            df.to_csv(dest_file,index=False)

                dest_dir = os.path.join(save_path,'Text')
                os.makedirs(dest_dir, exist_ok=True)                            
                dest_file = os.path.join(dest_dir,filename+".txt")
                #all_text = clean_text(all_text)
                with open(dest_file,'w') as dfil:
                    dfil.write(all_text)
                            
        else:
            dest_dir = os.path.join(save_path,'Images')
            os.makedirs(dest_dir, exist_ok=True)
            dest_file = os.path.join(dest_dir,file.split(os.sep)[-1])
            shutil.copy2(file, dest_file)
            

In [4]:
pdf_path = 'Documents For RAG'+os.sep+'Laws of Cricket'

extractData(pdf_path)


Document:  Document('Documents For RAG\Laws of Cricket\full-explanation_changes-to-the-laws-of-cricket-in-2022_v2_2.pdf')
Number of pages:  11
Document:  Document('Documents For RAG\Laws of Cricket\MCC_Laws_of_Cricket-2017-code-3rd-edition-2022.pdf')
Number of pages:  83
Document:  Document('Documents For RAG\Laws of Cricket\Tom Smiths Cricket Umpiring and Scoring-2017_code_version_2019.pdf')
Number of pages:  404
Document:  Document('Documents For RAG\Laws of Cricket\Aids to help\L24_Flow_Chart.pdf')
Number of pages:  1
Document:  Document('Documents For RAG\Laws of Cricket\Aids to help\L42_Process.pdf')
Number of pages:  1
Document:  Document('Documents For RAG\Laws of Cricket\Aids to help\LBW_Flow.pdf')
Number of pages:  1
Document:  Document('Documents For RAG\Laws of Cricket\Aids to help\Penalty_Runs_2022.pdf')
Number of pages:  1
Document:  Document('Documents For RAG\Laws of Cricket\Explanations\MCC Clarification on Scotland v Australia Men's T20I Incident. _ Lord's.pdf')
Number

In [5]:
def clean_text(text):    
    remove_text=rf"Laws of Cricket 2017 Code \(3rd Edition - 2022\)"
    text = re.sub(remove_text, " ",text)    

    text = text.replace("\\n\\d+"," ")

    # remove \n
    text = text.replace("\\n"," ")
    text = text.replace("\\t"," ")
    text = text.replace("\\r"," ")
    text = text.replace("\\b"," ")
    #remove \n \t \x \b
    #text = re.sub(r"[\n\t\b]"," ",text)
    

    text = re.sub(r"x\w+","",text)
    # remove special characters and punctuations
    text = re.sub(r"[^\w\s\.]"," ",text)

    #text =" ".join(text.split("\n"))

    #text = re.sub(r"n\w+","\w+",text)
    
    #text = re.sub(r"\n+"," ",text)
    #text = re.sub(r"\\n"," ",text)
    #text ="n ".join(text.split("n"))
    # remove single letters
    text = re.sub(r"\b[a-zA-Z]\b"," ",text)
    
    #text =" ".join(text.split("\\n"))
    # remove known unrequired text
    
    # remove html tags
    text = re.sub(r"<[^>]*>", " ",text)
    # lower case
    text = text.lower()
    # remove extra white space
    text = re.sub(r"\s+", " ",text)

    text = re.sub(r"\s\d ","",text)
    #trim trailing and leading edges
    text = text.strip()

    return text

In [6]:
aa="""such as\\nscoring runs and taking wickets, remain unchanged in such regulations.\\nThe Laws contained in this book are correct at the time of its publication but the MCC\\nwebsite (www.lords.org) and Laws of Cricket App provide a digital version which will be\\nupdated with any minor changes, if necessary.\\nSignificant dates in the history of the Laws are as follows:\\n1700 Cricket was recognised as early as this date.\\n1744 The earliest known Code was drawn up by certain \\xe2\\x80\\x9cNoblemen and Gentlemen\\xe2\\x80\\x9d who\\nused the Artillery Ground in London.\\n1755 The Laws were revised by \\xe2\\x80\\x9cSeveral Cricket Clubs, particularly the Star and Garter in \\nPall Mall\\xe2\\x80\\x9d.\\n'"b"b'Laws of Cricket 2017 Code (3rd Edition - 2022)\\n3\\n1774 A further revision was produced by \\xe2\\x80\\x9ca Committee of Noblemen and Gentlemen """

In [7]:
print(aa)

such as\nscoring runs and taking wickets, remain unchanged in such regulations.\nThe Laws contained in this book are correct at the time of its publication but the MCC\nwebsite (www.lords.org) and Laws of Cricket App provide a digital version which will be\nupdated with any minor changes, if necessary.\nSignificant dates in the history of the Laws are as follows:\n1700 Cricket was recognised as early as this date.\n1744 The earliest known Code was drawn up by certain \xe2\x80\x9cNoblemen and Gentlemen\xe2\x80\x9d who\nused the Artillery Ground in London.\n1755 The Laws were revised by \xe2\x80\x9cSeveral Cricket Clubs, particularly the Star and Garter in \nPall Mall\xe2\x80\x9d.\n'"b"b'Laws of Cricket 2017 Code (3rd Edition - 2022)\n3\n1774 A further revision was produced by \xe2\x80\x9ca Committee of Noblemen and Gentlemen 


In [8]:
bb=clean_text(aa)
bb

'such as scoring runs and taking wickets remain unchanged in such regulations. the laws contained in this book are correct at the time of its publication but the mcc website www.lords.org and laws of cricket app provide digital version which will be updated with any minor changes if necessary. significant dates in the history of the laws are as follows 1700 cricket was recognised as early as this date. 1744 the earliest known code was drawn up by certain and gentlemen who used the artillery ground in london. 1755 the laws were revised by cricket clubs particularly the star and garter in pall mall .1774 further revision was produced by committee of noblemen and gentlemen'

In [9]:
text_process_path  = r"RAG_prep\Laws of Cricket\Text"

In [10]:
text_process_path  = r"RAG_prep\Laws of Cricket\Text"

def clean_all_text(text_process_path):
    files_list = getFilesList(text_process_path)

    for file in files_list:
        with open(file,'r') as doc:
            clean_ver = clean_text(doc.read())
            if len(clean_ver)>0:
                out_dir = os.path.join(text_process_path,'Processed')
                os.makedirs(out_dir,exist_ok=True)
                out_file = out_dir+os.sep+file.split(os.sep)[-1]
                with open(out_file,'w') as of:
                    of.write(clean_ver)

clean_all_text(text_process_path)
    

In [27]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

def split_text_into_chunks(text_path,chunk_size=1000,chunk_overlap=0):
    files_list = getFilesList(text_path)    
    
    all_chunks = []
    text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size, 
      chunk_overlap=int(chunk_overlap*chunk_size/100), 
      length_function=len 
    ) 

    for file in files_list:
        loader = TextLoader(file)
        documents = loader.load()
        all_chunks.extend(text_splitter.split_documents(documents))

    return all_chunks
    

In [28]:
text_path = os.path.join(text_process_path,"Processed")#  = r"RAG_prep\Laws of Cricket\Text"
txt_chunks = split_text_into_chunks(text_path,chunk_size=4000,chunk_overlap=20)

In [29]:
print(len(txt_chunks))

for k in range(5):
    print(txt_chunks[k].metadata)

print(txt_chunks[-1])

299
{'source': 'RAG_prep\\Laws of Cricket\\Text\\Processed\\full-explanation_changes-to-the-laws-of-cricket-in-2022_v2_2.txt'}
{'source': 'RAG_prep\\Laws of Cricket\\Text\\Processed\\full-explanation_changes-to-the-laws-of-cricket-in-2022_v2_2.txt'}
{'source': 'RAG_prep\\Laws of Cricket\\Text\\Processed\\full-explanation_changes-to-the-laws-of-cricket-in-2022_v2_2.txt'}
{'source': 'RAG_prep\\Laws of Cricket\\Text\\Processed\\full-explanation_changes-to-the-laws-of-cricket-in-2022_v2_2.txt'}
{'source': 'RAG_prep\\Laws of Cricket\\Text\\Processed\\full-explanation_changes-to-the-laws-of-cricket-in-2022_v2_2.txt'}
page_content='must also with or without protective coverings permitted in law 5.4 be able to pass through bat gauge the dimensions and shape of which are shown in the diagram on the following page. bat gauge diagram .8 dimensions of aperture total depth 2.68 in 6.8 cm width 4.33 in 11.0 cm edge 161 in 41 cm curve 0.20 in 0.5 cm note the curve of the lower edge of the aperture is

In [51]:
print(txt_chunks[10].page_content)

law 24 flow chart time event current time penalty comments 11 00 the match start starts.mins 11 45 fielder smith tells the umpires he has pulled muscle in his leg and leaves the .mins this is an internal injury meaning smith is unable to bowl or bat until he has been back on the for the same length of time the penalty time he is off it. 13 00 scheduled time for lunch interval. 75 mins 13 40 the ne session of play commences. 75 mins lunch is scheduled interval no additional penalty time is added. 13 55 smith remains off the . 90 mins this is the ma penalty time no further time will be added. 14 15 play is suspended due to rain. 90 mins as smith has not told the umpires he is ready to resume the time of the unscheduled interruption is not taken off the penalty time. 14 45 play recommences but smith remains off the . 90 mins 15 00 play is suspended due to rain. 90 mins 15 10 smith advised the umpires he is ready to resume play. 90 mins 15 20 play recommences with smith on the of play. 80 

In [ ]:
# store text chunks into chroma db 
from langchain_huggingface import HuggingFaceEmbeddings # huggingface
from langchain.vectorstores import Chroma # chroma vector db


#def store_data_in_vector_db(all_chunks):    
# Initialize the Gemini embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2") 

chroma_path = 'VectorDB'
# Create -*=pothe Chroma database
chroma_db = Chroma.from_documents(
  collection_name = 'Laws of Cricket'
  documents=txt_chunks, 
  embedding=embeddings, 
  persist_directory=chroma_path
  collection_metadata={"hnsw:space": "cosine"}
)
chroma_db

# Persist the database to disk
#chroma_db.persist()


In [74]:
print(type(chroma_db))

<class 'langchain_community.vectorstores.chroma.Chroma'>


In [77]:
#from langchain.chains import SimilaritySearch

# Create similarity search chain
#similarity_search = SimilaritySearch(embeddings=embeddings)

# Query and get results with scores
query = "What conditions must be satisfied for a leg bye. Use the law, law changes, and Tom Smith"
docs = chroma_db.similarity_search_with_relevance_scores(query,k=10)
print(docs[0][0].page_content)

and make good their ground from end to end does not apply if other laws deem that run not to have been scored although the batsmen are not returned to their original ends. these laws are one or both batsmen run short law 18.4 batsman is dismissed caught law 33.4 striker is dismissed obstructing ball from being caught law 37.5 2. runs are disallowed when breach of law by one or both batsmen requires the umpire to cancel any completed runs and return the batsmen to their original ends in the following situations either batsman deliberately runs short law 18.5 leg byes are not awarded as the striker was neither attempting to play the ball with the bat nor avoiding injury law 23.3 an injured striker with runner is himself herself dismissed run out law 25.6.5 batsman runner leaves his her ground before the ball reaches the striker or passes the popping crease law 25.7 the batsmen run after ball has been lawfully struck more than once law 34.4 second or subsequent instance of any batsman


In [ ]:
for i,doc in enumerate(docs):
    #print(doc.metadata['source'])
    if(i%2==0):
        print(doc[0].page_content)
        print('Similarity Score: ',doc[1])

and make good their ground from end to end does not apply if other laws deem that run not to have been scored although the batsmen are not returned to their original ends. these laws are one or both batsmen run short law 18.4 batsman is dismissed caught law 33.4 striker is dismissed obstructing ball from being caught law 37.5 2. runs are disallowed when breach of law by one or both batsmen requires the umpire to cancel any completed runs and return the batsmen to their original ends in the following situations either batsman deliberately runs short law 18.5 leg byes are not awarded as the striker was neither attempting to play the ball with the bat nor avoiding injury law 23.3 an injured striker with runner is himself herself dismissed run out law 25.6.5 batsman runner leaves his her ground before the ball reaches the striker or passes the popping crease law 25.7 the batsmen run after ball has been lawfully struck more than once law 34.4 second or subsequent instance of any batsman
Sim

: 

In [52]:
docs

[(Document(metadata={'source': 'RAG_prep\\Laws of Cricket\\Text\\Processed\\Tom Smiths Cricket Umpiring and Scoring-2017_code_version_2019.txt'}, page_content='and make good their ground from end to end does not apply if other laws deem that run not to have been scored although the batsmen are not returned to their original ends. these laws are one or both batsmen run short law 18.4 batsman is dismissed caught law 33.4 striker is dismissed obstructing ball from being caught law 37.5 2. runs are disallowed when breach of law by one or both batsmen requires the umpire to cancel any completed runs and return the batsmen to their original ends in the following situations either batsman deliberately runs short law 18.5 leg byes are not awarded as the striker was neither attempting to play the ball with the bat nor avoiding injury law 23.3 an injured striker with runner is himself herself dismissed run out law 25.6.5 batsman runner leaves his her ground before the ball reaches the striker or

In [54]:
docs[0][0].page_content ==  docs[2][0].page_content

False